# 6: LLM-as-a-Judge Evaluation
## Faithfulness + Response Relevancy Scoring
## RAG-Based Medical QA System — PubMedQA
---
**What this notebook does:**
- Implements Faithfulness evaluation (Claim Extraction + Verification)
- Implements Response Relevancy evaluation (Alternate Query Generation + Cosine Similarity)
- Runs evaluation on all test queries from Step 5
- Produces a full evaluation report table for your report
- Shows detailed claim-level results for 3 example queries

## 6.1 Install Libraries

In [ ]:
!pip install groq sentence-transformers pandas tqdm -q
print('Done!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 14.0 MB/s eta 0:00:00
Done!


## 6.2 Imports & API Keys

In [ ]:
import json
import time
import re
import numpy as np
import pandas as pd
from groq import Groq
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
#  FILL IN YOUR GROQ KEY
# ============================================================
GROQ_API_KEY  = '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'
GROQ_MODEL    = 'llama-3.3-70b-versatile'
# ============================================================

groq_client = Groq(api_key=GROQ_API_KEY)
print('Groq client ready ✅')

Groq client ready ✅


## 6.3 Upload & Load RAG Results from Step 5

In [ ]:
from google.colab import files
print('Upload rag_results.json')
uploaded = files.upload()

Upload rag_results.json


Saving rag_results.json to rag_results.json


In [ ]:
with open('rag_results.json', 'r', encoding='utf-8') as f:
    rag_results = json.load(f)

print(f'Loaded {len(rag_results)} RAG results')
for i, r in enumerate(rag_results):
    print(f'  [{i+1}] {r["query"]}')

Loaded 5 RAG results
  [1] Does aspirin reduce cardiovascular risk?
  [2] What is the effect of metformin on type 2 diabetes?
  [3] Does exercise reduce breast cancer risk?
  [4] Are statins effective for lowering LDL cholesterol?
  [5] Is cognitive behavioral therapy effective for depression?


## 6.4 Load Embedding Model for Relevancy Scoring

In [ ]:
print('Loading sentence embedding model...')
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding model loaded ✅')

Loading sentence embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded ✅


## 6.5 Faithfulness Evaluation

**How it works (per assignment spec):**
1. Extract factual claims from the generated answer using LLM
2. Verify each claim against the retrieved context using LLM
3. Faithfulness Score = (supported claims / total claims) × 100%

**Prompt design:**
- Claim extraction prompt: asks LLM to list atomic facts from the answer
- Verification prompt: asks LLM to judge if each claim is supported by context

In [ ]:
def extract_claims(answer, groq_client, model=GROQ_MODEL):
    """
    Step 1 of Faithfulness: Extract atomic factual claims from the answer.

    Returns:
        list of claim strings
    """
    prompt = f"""You are an expert at extracting factual claims from medical text.

Given the following answer, extract all distinct atomic factual claims.
Each claim should be a single, verifiable statement.
Return ONLY a numbered list of claims, one per line.
Do not include opinions, hedges, or meta-statements about sources.

ANSWER:
{answer}

CLAIMS (numbered list):"""

    response = groq_client.chat.completions.create(
        model    = model,
        messages = [{'role': 'user', 'content': prompt}],
        max_tokens  = 300,
        temperature = 0.0
    )

    raw = response.choices[0].message.content.strip()

    # Parse numbered list into individual claims
    claims = []
    for line in raw.split('\n'):
        line = line.strip()
        # Remove numbering like "1." or "1)" from start
        cleaned = re.sub(r'^\d+[\.\)\:]\s*', '', line).strip()
        if cleaned and len(cleaned) > 10:  # filter empty/too-short lines
            claims.append(cleaned)

    return claims


def verify_claim(claim, context, groq_client, model=GROQ_MODEL):
    """
    Step 2 of Faithfulness: Verify a single claim against retrieved context.

    Returns:
        verdict : 'SUPPORTED' or 'NOT SUPPORTED'
        reason  : brief explanation
    """
    prompt = f"""You are a medical fact-checker. Your job is to determine if a claim is supported by the given context.

CONTEXT:
{context}

CLAIM:
{claim}

Is this claim directly supported by the context above?
Answer with EXACTLY one of: SUPPORTED or NOT SUPPORTED
Then on the next line, give a brief one-sentence reason.

FORMAT:
VERDICT: [SUPPORTED or NOT SUPPORTED]
REASON: [one sentence]"""

    response = groq_client.chat.completions.create(
        model    = model,
        messages = [{'role': 'user', 'content': prompt}],
        max_tokens  = 100,
        temperature = 0.0
    )

    raw = response.choices[0].message.content.strip()

    # Parse verdict
    verdict = 'NOT SUPPORTED'
    reason  = raw

    for line in raw.split('\n'):
        if 'VERDICT:' in line.upper():
            if 'NOT SUPPORTED' in line.upper():
                verdict = 'NOT SUPPORTED'
            elif 'SUPPORTED' in line.upper():
                verdict = 'SUPPORTED'
        if 'REASON:' in line.upper():
            reason = line.split(':', 1)[-1].strip()

    return verdict, reason


def evaluate_faithfulness(query, answer, retrieved_chunks, groq_client):
    """
    Full faithfulness evaluation for one query.

    Returns dict with:
        claims           : list of extracted claims
        verifications    : list of {claim, verdict, reason}
        faithfulness_score: float (0-100)
    """
    # Build full context string from retrieved chunks
    context = '\n\n'.join([
        f'[{i+1}] {chunk["text"]}'
        for i, chunk in enumerate(retrieved_chunks)
    ])

    # Step 1: Extract claims
    claims = extract_claims(answer, groq_client)

    if not claims:
        return {
            'claims'            : [],
            'verifications'     : [],
            'faithfulness_score': 0.0
        }

    # Step 2: Verify each claim
    verifications = []
    supported = 0

    for claim in claims:
        verdict, reason = verify_claim(claim, context, groq_client)
        verifications.append({
            'claim'  : claim,
            'verdict': verdict,
            'reason' : reason
        })
        if verdict == 'SUPPORTED':
            supported += 1
        time.sleep(0.3)  # avoid Groq rate limit

    score = round((supported / len(claims)) * 100, 1)

    return {
        'claims'            : claims,
        'verifications'     : verifications,
        'supported_count'   : supported,
        'total_claims'      : len(claims),
        'faithfulness_score': score
    }


print('Faithfulness functions defined ✅')

Faithfulness functions defined ✅


## 6.6 Response Relevancy Evaluation

**How it works:**
1. Generate 3 questions from the generated answer using LLM
2. Compute cosine similarity between each generated question and the original query
3. Relevancy Score = mean of 3 similarity scores × 100%

In [ ]:
def generate_questions_from_answer(answer, groq_client, model=GROQ_MODEL, n=3):
    """
    Step 1 of Relevancy: Generate N questions that the answer addresses.

    Returns:
        list of question strings
    """
    prompt = f"""You are a medical question generation expert.

Given the following answer from a medical research system, generate exactly {n} different questions that this answer could be responding to.
The questions should be natural, diverse, and directly answerable by the given answer.
Return ONLY a numbered list of questions, one per line.

ANSWER:
{answer}

GENERATED QUESTIONS:"""

    response = groq_client.chat.completions.create(
        model    = model,
        messages = [{'role': 'user', 'content': prompt}],
        max_tokens  = 200,
        temperature = 0.3  # slight creativity for diversity
    )

    raw = response.choices[0].message.content.strip()

    # Parse numbered list
    questions = []
    for line in raw.split('\n'):
        line = line.strip()
        cleaned = re.sub(r'^\d+[\.\)\:]\s*', '', line).strip()
        if cleaned and len(cleaned) > 10 and '?' in cleaned:
            questions.append(cleaned)
        if len(questions) == n:
            break

    # Fallback: take first n non-empty lines if no '?' found
    if len(questions) < n:
        questions = []
        for line in raw.split('\n'):
            cleaned = re.sub(r'^\d+[\.\)\:]\s*', '', line.strip()).strip()
            if cleaned and len(cleaned) > 10:
                questions.append(cleaned)
            if len(questions) == n:
                break

    return questions[:n]


def evaluate_relevancy(query, answer, embedder, groq_client):
    """
    Full relevancy evaluation for one query.

    Returns dict with:
        generated_questions : 3 questions from the answer
        similarities        : cosine similarity of each question to original query
        relevancy_score     : mean similarity × 100
    """
    # Step 1: Generate 3 questions from the answer
    generated_questions = generate_questions_from_answer(answer, groq_client, n=3)

    if not generated_questions:
        return {
            'generated_questions': [],
            'similarities'       : [],
            'relevancy_score'    : 0.0
        }

    # Step 2: Compute cosine similarity with original query
    query_emb     = embedder.encode([query], normalize_embeddings=True)
    question_embs = embedder.encode(generated_questions, normalize_embeddings=True)

    similarities = cosine_similarity(query_emb, question_embs)[0].tolist()
    similarities = [round(s, 4) for s in similarities]

    # Step 3: Mean similarity as relevancy score
    relevancy_score = round(np.mean(similarities) * 100, 1)

    return {
        'generated_questions': generated_questions,
        'similarities'       : similarities,
        'relevancy_score'    : relevancy_score
    }


print('Relevancy functions defined ✅')

Relevancy functions defined ✅


## 6.7 Run Full Evaluation on All 5 Queries
This runs both Faithfulness and Relevancy for every query.

In [ ]:
evaluation_results = []

for i, result in enumerate(rag_results):
    query     = result['query']
    answer    = result['answer']
    retrieved = result['retrieved']

    print(f'\n{"-"*65}')
    print(f'[{i+1}/5] Evaluating: {query}')
    print(f'{"-"*65}')

    # --- Faithfulness ---
    print('  Running faithfulness evaluation...')
    faith_result = evaluate_faithfulness(query, answer, retrieved, groq_client)
    print(f'  Claims found    : {faith_result["total_claims"]}')
    print(f'  Claims supported: {faith_result["supported_count"]}')
    print(f'  Faithfulness    : {faith_result["faithfulness_score"]}%')

    time.sleep(0.5)

    # --- Relevancy ---
    print('  Running relevancy evaluation...')
    rel_result = evaluate_relevancy(query, answer, embedder, groq_client)
    print(f'  Generated questions: {len(rel_result["generated_questions"])}')
    print(f'  Similarities   : {rel_result["similarities"]}')
    print(f'  Relevancy score: {rel_result["relevancy_score"]}%')

    evaluation_results.append({
        'query'             : query,
        'answer'            : answer,
        'faithfulness'      : faith_result,
        'relevancy'         : rel_result
    })

    time.sleep(1.0)  # rate limit buffer

print(f'\n{"="*65}')
print('ALL 5 EVALUATIONS COMPLETE ✅')
print(f'{"="*65}')


-----------------------------------------------------------------
[1/5] Evaluating: Does aspirin reduce cardiovascular risk?
-----------------------------------------------------------------
  Running faithfulness evaluation...
  Claims found    : 7
  Claims supported: 6
  Faithfulness    : 85.7%
  Running relevancy evaluation...
  Generated questions: 3
  Similarities   : [0.896, 0.7864, 0.8038]
  Relevancy score: 82.9%

-----------------------------------------------------------------
[2/5] Evaluating: What is the effect of metformin on type 2 diabetes?
-----------------------------------------------------------------
  Running faithfulness evaluation...
  Claims found    : 4
  Claims supported: 4
  Faithfulness    : 100.0%
  Running relevancy evaluation...
  Generated questions: 3
  Similarities   : [0.7841, 0.9595, 0.8233]
  Relevancy score: 85.6%

-----------------------------------------------------------------
[3/5] Evaluating: Does exercise reduce breast cancer risk?
---------

## 6.8 Summary Table

In [ ]:
rows = []
for r in evaluation_results:
    rows.append({
        'Query'                : r['query'][:45] + '...',
        'Total Claims'         : r['faithfulness']['total_claims'],
        'Supported Claims'     : r['faithfulness']['supported_count'],
        'Faithfulness (%)'     : r['faithfulness']['faithfulness_score'],
        'Relevancy (%)'        : r['relevancy']['relevancy_score'],
    })

df = pd.DataFrame(rows)

# Add average row
avg = df.mean(numeric_only=True).round(1).to_dict()
avg['Query'] = 'AVERAGE'
df = pd.concat([df, pd.DataFrame([avg])], ignore_index=True)

print('=== EVALUATION SUMMARY TABLE (for your report) ===')
print(df.to_string(index=False))
print('\n(Copy this table into your report Section D: Performance Metrics)')

=== EVALUATION SUMMARY TABLE (for your report) ===
                                           Query  Total Claims  Supported Claims  Faithfulness (%)  Relevancy (%)
     Does aspirin reduce cardiovascular risk?...           7.0               6.0              85.7           82.9
What is the effect of metformin on type 2 dia...           4.0               4.0             100.0           85.6
     Does exercise reduce breast cancer risk?...           5.0               4.0              80.0           87.2
Are statins effective for lowering LDL choles...           7.0               7.0             100.0           87.6
Is cognitive behavioral therapy effective for...           8.0               8.0             100.0           83.6
                                         AVERAGE           6.2               5.8              93.1           85.4

(Copy this table into your report Section D: Performance Metrics)


## 6.9 Detailed Claim-Level Results for 3 Example Queries
**Required by assignment:** Show extracted claims and verification results for at least 3 queries.

In [ ]:
# Show detailed breakdown for first 3 queries
for i in range(min(3, len(evaluation_results))):
    r = evaluation_results[i]

    print(f'\n{"="*70}')
    print(f'QUERY {i+1}: {r["query"]}')
    print(f'{"="*70}')

    print(f'\nANSWER:')
    print(f'{r["answer"][:400]}...' if len(r['answer']) > 400 else r['answer'])

    # Faithfulness details
    print(f'\n--- FAITHFULNESS EVALUATION ---')
    print(f'Total Claims   : {r["faithfulness"]["total_claims"]}')
    print(f'Supported      : {r["faithfulness"]["supported_count"]}')
    print(f'Faithfulness   : {r["faithfulness"]["faithfulness_score"]}%')
    print(f'\nClaim-by-Claim Verification:')
    for j, v in enumerate(r['faithfulness']['verifications']):
        icon = '✅' if v['verdict'] == 'SUPPORTED' else '❌'
        print(f'  {icon} Claim {j+1}: {v["claim"]}')
        print(f'     Verdict : {v["verdict"]}')
        print(f'     Reason  : {v["reason"]}')

    # Relevancy details
    print(f'\n--- RELEVANCY EVALUATION ---')
    print(f'Original Query : {r["query"]}')
    print(f'\nGenerated Questions & Similarities:')
    for j, (q, s) in enumerate(zip(
        r['relevancy']['generated_questions'],
        r['relevancy']['similarities']
    )):
        print(f'  Q{j+1}: {q}')
        print(f'      Cosine Similarity: {s}')
    print(f'\n  Relevancy Score (mean): {r["relevancy"]["relevancy_score"]}%')

print(f'\n{"="*70}')
print('(Copy this section into your report as Appendix or Section D examples)')
print(f'{"="*70}')


QUERY 1: Does aspirin reduce cardiovascular risk?

ANSWER:
Yes, aspirin reduces cardiovascular risk. According to [2], aspirin is used in the secondary prevention of cardiovascular disease (CVD), and [3] shows that beta-blockers, which are often used in conjunction with aspirin, result in significant risk reductions for cardiovascular events, cardiovascular death, and stroke. Additionally, [1] mentions that antiplatelets such as aspirin are used to reduce...

--- FAITHFULNESS EVALUATION ---
Total Claims   : 7
Supported      : 6
Faithfulness   : 85.7%

Claim-by-Claim Verification:
  ✅ Claim 1: Aspirin reduces cardiovascular risk.
     Verdict : SUPPORTED
     Reason  : The context provides evidence from various studies that aspirin use is associated with reduced risk of cardiovascular events, including secondary prevention of cardiovascular disease and reduction of major adverse cardiac events.
  ✅ Claim 2: Aspirin is used in the secondary prevention of cardiovascular disease.
     Ver

## 6.10 Save Evaluation Results

In [ ]:
with open('evaluation_results.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_results, f, indent=2, ensure_ascii=False)
print('evaluation_results.json saved ✅')

# Also save summary table as CSV for easy copying
df.to_csv('evaluation_summary.csv', index=False)
print('evaluation_summary.csv saved ✅')

print('\nStep 6 Complete! ✅')
print('Files ready:')
print('  evaluation_results.json  → full detailed results')
print('  evaluation_summary.csv   → summary table for report')
print('\nNext: Step 7 — Ablation Study')

evaluation_results.json saved ✅
evaluation_summary.csv saved ✅

Step 6 Complete! ✅
Files ready:
  evaluation_results.json  → full detailed results
  evaluation_summary.csv   → summary table for report

Next: Step 7 — Ablation Study


## 6.11 Download Files

In [ ]:
from google.colab import files
files.download('evaluation_results.json')
files.download('evaluation_summary.csv')
print('Both files downloaded!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Both files downloaded!
